# Module 6: Performance Tuning

**Objective**: Optimize Spark jobs using broadcast joins, caching, partitioning, and Spark UI analysis.

## Key Topics
- Broadcast joins for small tables
- Caching strategies
- Partition tuning
- Spark UI analysis

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, count, sum as spark_sum
from pathlib import Path
import time

spark = SparkSession.builder.appName("Module06-Performance").master("local[*]").config("spark.sql.shuffle.partitions", "8").config("spark.sql.autoBroadcastJoinThreshold", "10MB").getOrCreate()
DATA_RAW = Path("../data/raw")
print(f"Spark UI: http://localhost:4040")

In [ ]:
branches_df = spark.read.csv(str(DATA_RAW / "branches.csv"), header=True, inferSchema=True)
accounts_df = spark.read.csv(str(DATA_RAW / "accounts.csv"), header=True, inferSchema=True)
transactions_df = spark.read.csv(str(DATA_RAW / "transactions.csv"), header=True, inferSchema=True)
print(f"Branches: {branches_df.count()}, Accounts: {accounts_df.count()}, Transactions: {transactions_df.count()}")

## 1. Broadcast Joins

In [ ]:
# Normal join (shuffle both sides)
start = time.time()
normal_join = accounts_df.join(branches_df, "branch_id")
normal_join.count()
print(f"Normal join: {time.time()-start:.2f}s")

# Broadcast join (broadcast small table)
start = time.time()
broadcast_join = accounts_df.join(broadcast(branches_df), "branch_id")
broadcast_join.count()
print(f"Broadcast join: {time.time()-start:.2f}s")

In [ ]:
# Compare execution plans
print("=== Normal Join Plan ===")
normal_join.explain()
print("\n=== Broadcast Join Plan ===")
broadcast_join.explain()

## 2. Caching Strategies

In [ ]:
# Without caching (reads data twice)
start = time.time()
result1 = transactions_df.filter(col("status") == "Completed").count()
result2 = transactions_df.filter(col("status") == "Failed").count()
print(f"Without cache: {time.time()-start:.2f}s")

# With caching
transactions_df.cache()
transactions_df.count()  # Materialize cache

start = time.time()
result1 = transactions_df.filter(col("status") == "Completed").count()
result2 = transactions_df.filter(col("status") == "Failed").count()
print(f"With cache: {time.time()-start:.2f}s")

transactions_df.unpersist()  # Clean up

## 3. Partition Tuning

In [ ]:
print(f"Current partitions: {transactions_df.rdd.getNumPartitions()}")

# Repartition (expensive - full shuffle)
txn_repartitioned = transactions_df.repartition(16)
print(f"After repartition(16): {txn_repartitioned.rdd.getNumPartitions()}")

# Coalesce (cheap - no shuffle, only reduces)
txn_coalesced = transactions_df.coalesce(4)
print(f"After coalesce(4): {txn_coalesced.rdd.getNumPartitions()}")

## 4. Shuffle Partition Tuning

In [ ]:
# Default is 200, too high for local
print(f"Current shuffle.partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

# Compare aggregation with different settings
for partitions in [4, 8, 16]:
    spark.conf.set("spark.sql.shuffle.partitions", str(partitions))
    start = time.time()
    transactions_df.groupBy("channel").agg(count("*"), spark_sum("amount")).collect()
    print(f"Partitions={partitions}: {time.time()-start:.2f}s")

## 5. Check Spark UI

Open http://localhost:4040 and explore:
- **Jobs**: See all executed jobs
- **Stages**: Drill into shuffle read/write
- **Storage**: View cached DataFrames
- **SQL**: See query execution plans

## Performance Checklist

✅ Use `broadcast()` for joins with small tables (<100MB)
✅ Cache DataFrames reused multiple times
✅ Reduce `shuffle.partitions` for local dev (4-16)
✅ Use `coalesce()` instead of `repartition()` to reduce partitions
✅ Filter early to reduce data volume
✅ Use columnar formats (Parquet) over CSV

In [ ]:
spark.stop()